# Exploratory analysis — F1 Race Intelligence
Executed against the processed dataset produced by `python -m src.data_pipeline`. All numbers below are computed, not typed in. Correlations are descriptive, not causal.

In [1]:
import sys, json
sys.path.insert(0, '..')
import pandas as pd
from src import eda
from src.config import resolve
df = pd.read_csv(resolve('data/processed/model_dataset.csv'))
study = pd.read_csv(resolve('data/processed/features_study.csv'))
print(df.shape, study.shape)

(3910, 57) (4595, 57)


## Data quality

In [2]:
q = eda.data_quality(df)
q[q['missing'] > 0]

,dtype,missing,missing_pct,unique
quali_position,float64,10,0.26,22
exclusion_reason,float64,3910,100.00,0
drv_avg_pos_3,float64,30,0.77,71
drv_avg_pos_5,float64,30,0.77,132
drv_podiums_5,float64,30,0.77,6
drv_ppr_5,float64,30,0.77,125
drv_dnf_rate_5,float64,30,0.77,11
drv_avg_grid_5,float64,30,0.77,137
con_avg_pos_3,float64,1,0.03,110
con_avg_pos_5,float64,1,0.03,176


In [3]:
print('duplicate driver-race keys:', df.duplicated(['season','round','driver_id']).sum())
json.load(open(resolve('data/processed/cleaning_summary.json')))['exclusions_by_reason']

duplicate driver-race keys: 0


[{'reason': 'retired (not classified)', 'rows': 676},
 {'reason': 'withdrawn / did not start', 'rows': 31},
 {'reason': 'disqualified', 'rows': 9}]

## Class balance by chronological split

In [4]:
eda.class_balance(df)

,rows,podiums,non_podiums,podium_rate
split,,,,
train,2324,414,1910,0.1781
validation,764,132,632,0.1728
test,822,138,684,0.1679


## Grid slot and podium rate

In [5]:
eda.podium_rate_by_grid(df).round(3)

,grid_clean,rate,n
0,1.0,0.880,209
1,2.0,0.759,212
2,3.0,0.595,210
3,4.0,0.382,204
4,5.0,0.193,197
5,6.0,0.160,206
6,7.0,0.074,202
7,8.0,0.037,189
8,9.0,0.036,192
9,10.0,0.049,185


## Experience and missing history

In [6]:
eda.experience_vs_podium(df)

,exp_bin,rate,n
0,rookie (0),0.033333,30
1,1-10,0.003759,266
2,11-25,0.014286,350
3,26-50,0.089027,483
4,51-100,0.112038,839
5,101-150,0.271132,627
6,151-200,0.356998,493
7,200+,0.236010,822


In [7]:
eda.missing_history(study)

{'rookie_entries': 38,
 'returning_entries': 18,
 'team_change_entries': 53,
 'teams_without_history_entries': 2,
 'missing_by_feature': {'drv_avg_pos_3': 38,
  'drv_avg_pos_5': 38,
  'drv_podiums_5': 38,
  'drv_ppr_5': 38,
  'drv_dnf_rate_5': 38,
  'drv_avg_grid_5': 38,
  'con_avg_pos_3': 2,
  'con_avg_pos_5': 2,
  'con_podiums_5': 2,
  'con_ppr_5': 2,
  'drv_has_history': 0,
  'con_has_history': 0,
  'drv_gap_races': 38,
  'drv_career_avg_pos': 38,
  'drv_career_podium_rate': 38}}

## Circuits where podium finishers start furthest back

In [8]:
eda.circuit_patterns(df).head(10)

,circuit_name,races,mean_podium_grid,podiums_from_outside_top6,share_outside_top6
0,Hockenheimring,4,7.250000,4,0.333
1,Hungaroring,11,4.484848,8,0.242
2,Autódromo José Carlos Pace,10,4.333333,5,0.167
3,Baku City Circuit,8,4.083333,5,0.208
4,Circuit de Spa-Francorchamps,11,3.787879,4,0.121
5,Circuit of the Americas,10,3.733333,3,0.100
6,Autodromo Nazionale di Monza,11,3.545455,4,0.121
7,Red Bull Ring,13,3.512821,6,0.154
8,Circuit Park Zandvoort,4,3.500000,1,0.083
9,Sochi Autodrom,8,3.500000,3,0.125


## Spearman correlation with the target

In [9]:
eda.correlation(df)['podium'].sort_values()

grid_clean               -0.557
con_avg_pos_5            -0.528
drv_avg_pos_5            -0.516
drv_career_starts         0.248
drv_races_with_con        0.273
con_season_pts_before     0.423
drv_season_pts_before     0.433
drv_career_podium_rate    0.442
con_ppr_5                 0.545
drv_ppr_5                 0.546
drv_podiums_5             0.592
podium                    1.000
Name: podium, dtype: float64

## Podium rate by constructor recent points-per-race decile

In [10]:
eda.feature_target_deciles(df, 'con_ppr_5').round(3)

,mid,rate,n
0,0.0,0.002,805
1,1.4,0.005,401
2,2.6,0.025,364
3,4.6,0.017,423
4,7.4,0.046,369
5,11.8,0.122,376
6,17.4,0.329,392
7,24.4,0.499,389
8,34.4,0.713,390
